# AuraGateway governed full A/B/C environment qualification

Cold-session launcher for one authorized six-probe qualification. No benchmark trajectory is permitted. Use T4 x2, Internet Off, no secrets, and Save Version -> Save & Run All.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import socket
import sys
import traceback
import zipfile
from datetime import UTC, datetime
from pathlib import Path

NOTEBOOK_NAME = "ag-full-abc-env-qualification-v1"
INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()
MATERIALIZED_HARNESS_ROOT = WORK_ROOT / "auragateway_qualification_harness"
EVIDENCE_ZIP_PATH = WORK_ROOT / "ag-qualification-evidence-v1.zip"

CONTROL_NOTEBOOK_TOKEN = "ag-qualification-control-materializer-v1"
CONTROL_MANIFEST_NAME = "control_package_manifest.json"
CONTROL_RECEIPT_NAME = "materialization_receipt.json"
AUTHORIZATION_FILENAME = (
    "auragateway_full_abc_local_full_run_environment_qualification_execution_auth"
    "orization_v1.json"
)
DATASET_MANIFEST_FILENAME = "offline_dataset_manifest.json"

EXPECTED_SOURCE_MAIN_MERGE_COMMIT = "99aed79c582d1fb5cfceaa96840ea97433f62423"
EXPECTED_AUTHORIZATION_SOURCE_MAIN_MERGE_COMMIT = (
    "211a10757999b1b110cb1d9df172938cf6ed7969"
)
EXPECTED_REVIEWED_CORE_SHA256 = "b5f9b3ab7534f3a17b6b82c214edbac8bdcce650b4572721abdb054c35d0c613"
REVIEWED_CORE_B64 = (
    "aW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1w"
    "b3J0IHNodXRpbAppbXBvcnQgc3RhdAppbXBvcnQgc3lzCmZyb20gZGF0ZXRpbWUgaW1wb3J0IFVU"
    "QywgZGF0ZXRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpFWFBFQ1RFRF9SRVFVRVNUX1NI"
    "QTI1NiA9ICgKICAgICJkY2VmN2U3MjQzZjRkZTE2OTU1YmNjZGZjMzZkYmQwMTk0YjUxYTYwMmQx"
    "ZmM2N2Y1YzZmYTM3NWNhNTI5ZTI4IgopCkVYUEVDVEVEX1JVTlRJTUVfRkFDVE9SWSA9ICgKICAg"
    "ICJhdXJhZ2F0ZXdheS5sb2NhbF9hYmMuIgogICAgImZ1bGxfYWJjX2xvY2FsX2Vudmlyb25tZW50"
    "X3F1YWxpZmljYXRpb25fa2FnZ2xlX3J1bnRpbWVfYWRhcHRlcjoiCiAgICAiY3JlYXRlX3J1bnRp"
    "bWVfYWRhcHRlciIKKQpLQUdHTEVfSU5QVVRfUk9PVCA9IFBhdGgoIi9rYWdnbGUvaW5wdXQiKS5y"
    "ZXNvbHZlKCkKS0FHR0xFX1dPUktJTkdfUk9PVCA9IFBhdGgoIi9rYWdnbGUvd29ya2luZyIpLnJl"
    "c29sdmUoKQpNQVRFUklBTElaRURfSEFSTkVTU19ST09UID0gS0FHR0xFX1dPUktJTkdfUk9PVCAv"
    "ICJhdXJhZ2F0ZXdheV9xdWFsaWZpY2F0aW9uX2hhcm5lc3MiCgoKZGVmIF9yZXF1aXJlZF9lbnZp"
    "cm9ubWVudChuYW1lKToKICAgIHZhbHVlID0gb3MuZW52aXJvbi5nZXQobmFtZSkKICAgIGlmIHZh"
    "bHVlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYicmVxdWlyZWQgZW52aXJv"
    "bm1lbnQgYmluZGluZyBpcyBtaXNzaW5nOiB7bmFtZX0iKQogICAgcmV0dXJuIHZhbHVlCgoKZGVm"
    "IF9ib3VuZGVkX2lucHV0X3BhdGgocmF3X3BhdGgsICosIGV4cGVjdGVkX2tpbmQpOgogICAgcGF0"
    "aCA9IFBhdGgocmF3X3BhdGgpLnJlc29sdmUoKQogICAgaWYgcGF0aCAhPSBLQUdHTEVfSU5QVVRf"
    "Uk9PVCBhbmQgS0FHR0xFX0lOUFVUX1JPT1Qgbm90IGluIHBhdGgucGFyZW50czoKICAgICAgICBy"
    "YWlzZSBSdW50aW1lRXJyb3IoZiJ7ZXhwZWN0ZWRfa2luZH0gbXVzdCByZW1haW4gdW5kZXIgL2th"
    "Z2dsZS9pbnB1dCIpCiAgICByZXR1cm4gcGF0aAoKCmRlZiBfZmlsZV9zaGEyNTYocGF0aCk6CiAg"
    "ICBpZiBub3QgcGF0aC5pc19maWxlKCkgb3IgcGF0aC5pc19zeW1saW5rKCk6CiAgICAgICAgcmFp"
    "c2UgUnVudGltZUVycm9yKCJleHBlY3RlZCBvbmUgcmVndWxhciBmaWxlIikKICAgIGRpZ2VzdCA9"
    "IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIGhhbmRsZToKICAg"
    "ICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGhhbmRsZS5yZWFkKDEwMjQgKiAxMDI0KSwg"
    "YiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdlc3Qu"
    "aGV4ZGlnZXN0KCkKCgpkZWYgX2Nhbm9uaWNhbF9zaGEyNTYocGF5bG9hZCk6CiAgICBjYW5vbmlj"
    "YWwgPSBqc29uLmR1bXBzKHBheWxvYWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgiLCIs"
    "ICI6IikpCiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoY2Fub25pY2FsLmVuY29kZSgidXRmLTgi"
    "KSkuaGV4ZGlnZXN0KCkKCgpkZWYgX2RpcmVjdG9yeV9zaGEyNTYocm9vdCk6CiAgICBlbnRyaWVz"
    "ID0gW10KICAgIHRvdGFsX2J5dGVzID0gMAogICAgZm9yIHBhdGggaW4gc29ydGVkKHJvb3Qucmds"
    "b2IoIioiKSwga2V5PWxhbWJkYSBpdGVtOiBpdGVtLmFzX3Bvc2l4KCkpOgogICAgICAgIGlmIHBh"
    "dGguaXNfc3ltbGluaygpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImhhcm5lc3Mg"
    "c291cmNlIGNvbnRhaW5zIGEgc3ltYm9saWMgbGluayIpCiAgICAgICAgbWV0YWRhdGEgPSBwYXRo"
    "LnN0YXQoKQogICAgICAgIGlmIHBhdGguaXNfZGlyKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAg"
    "ICAgICAgaWYgbm90IHN0YXQuU19JU1JFRyhtZXRhZGF0YS5zdF9tb2RlKToKICAgICAgICAgICAg"
    "cmFpc2UgUnVudGltZUVycm9yKCJoYXJuZXNzIHNvdXJjZSBjb250YWlucyBhIG5vbi1yZWd1bGFy"
    "IG1lbWJlciIpCiAgICAgICAgdG90YWxfYnl0ZXMgKz0gbWV0YWRhdGEuc3Rfc2l6ZQogICAgICAg"
    "IGlmIHRvdGFsX2J5dGVzID4gMTAwICogMTAyNCAqIDEwMjQ6CiAgICAgICAgICAgIHJhaXNlIFJ1"
    "bnRpbWVFcnJvcigiaGFybmVzcyBzb3VyY2UgZXhjZWVkcyB0aGUgYm9vdHN0cmFwIGJ5dGUgYnVk"
    "Z2V0IikKICAgICAgICBlbnRyaWVzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAg"
    "ICAgInBhdGgiOiBwYXRoLnJlbGF0aXZlX3RvKHJvb3QpLmFzX3Bvc2l4KCksCiAgICAgICAgICAg"
    "ICAgICAic2hhMjU2IjogX2ZpbGVfc2hhMjU2KHBhdGgpLAogICAgICAgICAgICAgICAgInNpemVf"
    "Ynl0ZXMiOiBtZXRhZGF0YS5zdF9zaXplLAogICAgICAgICAgICB9CiAgICAgICAgKQogICAgICAg"
    "IGlmIGxlbihlbnRyaWVzKSA+IDVfMDAwOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3Io"
    "Imhhcm5lc3Mgc291cmNlIGV4Y2VlZHMgdGhlIGJvb3RzdHJhcCBmaWxlIGJ1ZGdldCIpCiAgICBp"
    "ZiBub3QgZW50cmllczoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImhhcm5lc3Mgc291cmNl"
    "IGlzIGVtcHR5IikKICAgIHJldHVybiBfY2Fub25pY2FsX3NoYTI1NihlbnRyaWVzKQoKCmRlZiBf"
    "cGFyc2VfdGltZXN0YW1wKHJhd192YWx1ZSk6CiAgICBpZiBub3QgaXNpbnN0YW5jZShyYXdfdmFs"
    "dWUsIHN0cik6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJhdXRob3JpemF0aW9uIHRpbWVz"
    "dGFtcCBpcyBpbnZhbGlkIikKICAgIG5vcm1hbGl6ZWQgPSByYXdfdmFsdWUucmVwbGFjZSgiWiIs"
    "ICIrMDA6MDAiKQogICAgdmFsdWUgPSBkYXRldGltZS5mcm9taXNvZm9ybWF0KG5vcm1hbGl6ZWQp"
    "CiAgICBpZiB2YWx1ZS50emluZm8gaXMgTm9uZSBvciB2YWx1ZS51dGNvZmZzZXQoKSBpcyBOb25l"
    "OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiYXV0aG9yaXphdGlvbiB0aW1lc3RhbXAgbXVz"
    "dCBiZSB0aW1lem9uZS1hd2FyZSIpCiAgICByZXR1cm4gdmFsdWUKCgphdXRob3JpemF0aW9uX3Bh"
    "dGggPSBfYm91bmRlZF9pbnB1dF9wYXRoKAogICAgX3JlcXVpcmVkX2Vudmlyb25tZW50KCJBVVJB"
    "R0FURVdBWV9RVUFMSUZJQ0FUSU9OX0FVVEhPUklaQVRJT04iKSwKICAgIGV4cGVjdGVkX2tpbmQ9"
    "ImF1dGhvcml6YXRpb24iLAopCm1hbmlmZXN0X3BhdGggPSBfYm91bmRlZF9pbnB1dF9wYXRoKAog"
    "ICAgX3JlcXVpcmVkX2Vudmlyb25tZW50KCJBVVJBR0FURVdBWV9RVUFMSUZJQ0FUSU9OX0RBVEFT"
    "RVRfTUFOSUZFU1QiKSwKICAgIGV4cGVjdGVkX2tpbmQ9ImRhdGFzZXQgbWFuaWZlc3QiLAopCmF1"
    "dGhvcml6YXRpb24gPSBqc29uLmxvYWRzKGF1dGhvcml6YXRpb25fcGF0aC5yZWFkX3RleHQoZW5j"
    "b2Rpbmc9InV0Zi04IikpCm1hbmlmZXN0ID0ganNvbi5sb2FkcyhtYW5pZmVzdF9wYXRoLnJlYWRf"
    "dGV4dChlbmNvZGluZz0idXRmLTgiKSkKaWYgYXV0aG9yaXphdGlvbi5nZXQoImRlY2lzaW9uIikg"
    "IT0gIkFVVEhPUklaRUQiOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJhdXRob3JpemF0aW9uIGRl"
    "Y2lzaW9uIGlzIG5vdCBBVVRIT1JJWkVEIikKaWYgYXV0aG9yaXphdGlvbi5nZXQoInJlcXVlc3Rf"
    "c2hhMjU2IikgIT0gRVhQRUNURURfUkVRVUVTVF9TSEEyNTY6CiAgICByYWlzZSBSdW50aW1lRXJy"
    "b3IoImF1dGhvcml6YXRpb24gZG9lcyBub3QgYmluZCB0aGUgZXhwZWN0ZWQgcmVxdWVzdCIpCmlm"
    "IGF1dGhvcml6YXRpb24uZ2V0KCJkYXRhc2V0X21hbmlmZXN0X3NoYTI1NiIpICE9IF9jYW5vbmlj"
    "YWxfc2hhMjU2KG1hbmlmZXN0KToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiYXV0aG9yaXphdGlv"
    "biBkb2VzIG5vdCBiaW5kIHRoZSBzdXBwbGllZCBkYXRhc2V0IG1hbmlmZXN0IikKZW50cmllcyA9"
    "IG1hbmlmZXN0LmdldCgiZW50cmllcyIpCmV4cGVjdGVkX2VudHJ5X3NoYXBlcyA9ICgKICAgICgi"
    "aGFybmVzc19zb3VyY2UiLCAic291cmNlX3RyZWVfZGlyZWN0b3J5IiksCiAgICAoIm1vZGVsX2Fy"
    "dGlmYWN0cyIsICJodWdnaW5nX2ZhY2Vfc25hcHNob3RfZGlyZWN0b3J5IiksCiAgICAoInZsbG1f"
    "d2hlZWwiLCAicHl0aG9uX3doZWVsIiksCikKaWYgbm90IGlzaW5zdGFuY2UoZW50cmllcywgbGlz"
    "dCkgb3IgbGVuKGVudHJpZXMpICE9IGxlbihleHBlY3RlZF9lbnRyeV9zaGFwZXMpOgogICAgcmFp"
    "c2UgUnVudGltZUVycm9yKCJkYXRhc2V0IG1hbmlmZXN0IGVudHJ5IHNldCBkcmlmdGVkIikKZm9y"
    "IGVudHJ5LCBleHBlY3RlZF9zaGFwZSBpbiB6aXAoZW50cmllcywgZXhwZWN0ZWRfZW50cnlfc2hh"
    "cGVzLCBzdHJpY3Q9VHJ1ZSk6CiAgICBpZiBub3QgaXNpbnN0YW5jZShlbnRyeSwgZGljdCk6CiAg"
    "ICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJkYXRhc2V0IG1hbmlmZXN0IGVudHJ5IGlzIGludmFs"
    "aWQiKQogICAgb2JzZXJ2ZWRfc2hhcGUgPSAoZW50cnkuZ2V0KCJyb2xlIiksIGVudHJ5LmdldCgi"
    "YXJ0aWZhY3RfZm9ybWF0IikpCiAgICBpZiBvYnNlcnZlZF9zaGFwZSAhPSBleHBlY3RlZF9zaGFw"
    "ZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImRhdGFzZXQgbWFuaWZlc3Qgcm9sZSBvciBh"
    "cnRpZmFjdCBmb3JtYXQgZHJpZnRlZCIpCiAgICBfYm91bmRlZF9pbnB1dF9wYXRoKAogICAgICAg"
    "IGVudHJ5LmdldCgibW91bnRlZF9wYXRoIiksCiAgICAgICAgZXhwZWN0ZWRfa2luZD1mIntleHBl"
    "Y3RlZF9zaGFwZVswXX0gaW5wdXQiLAogICAgKQppZiBhbnkoCiAgICAoCiAgICAgICAgbWFuaWZl"
    "c3QuZ2V0KCJuZXR3b3JrX2FjY2Vzc19wZXJtaXR0ZWQiKSBpcyBub3QgRmFsc2UsCiAgICAgICAg"
    "bWFuaWZlc3QuZ2V0KCJjcmVkZW50aWFsc19wcmVzZW50IikgaXMgbm90IEZhbHNlLAogICAgICAg"
    "IG1hbmlmZXN0LmdldCgiY3VzdG9tZXJfZGF0YV9wcmVzZW50IikgaXMgbm90IEZhbHNlLAogICAg"
    "ICAgIG1hbmlmZXN0LmdldCgiaG9zdGVkX3Byb3ZpZGVyX2lucHV0c19wcmVzZW50IikgaXMgbm90"
    "IEZhbHNlLAogICAgKQopOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJkYXRhc2V0IG1hbmlmZXN0"
    "IHNhZmV0eSBlbnZlbG9wZSBkcmlmdGVkIikKaWYgYW55KAogICAgKAogICAgICAgIGF1dGhvcml6"
    "YXRpb24uZ2V0KCJtYXhpbXVtX2thZ2dsZV9zZXNzaW9ucyIpICE9IDEsCiAgICAgICAgYXV0aG9y"
    "aXphdGlvbi5nZXQoIm1heGltdW1fbW9kZWxfcmVxdWVzdHMiKSAhPSA4LAogICAgICAgIGF1dGhv"
    "cml6YXRpb24uZ2V0KCJtYXhpbXVtX291dHB1dF90b2tlbnNfcGVyX3JlcXVlc3QiKSAhPSAzMiwK"
    "ICAgICAgICBhdXRob3JpemF0aW9uLmdldCgiYmVuY2htYXJrX3RyYWplY3RvcnlfcmVxdWVzdHNf"
    "cGVybWl0dGVkIikgIT0gMCwKICAgICAgICBhdXRob3JpemF0aW9uLmdldCgiY3VzdG9tZXJfZGF0"
    "YV9wZXJtaXR0ZWQiKSBpcyBub3QgRmFsc2UsCiAgICAgICAgYXV0aG9yaXphdGlvbi5nZXQoImNy"
    "ZWRlbnRpYWxzX3Blcm1pdHRlZCIpIGlzIG5vdCBGYWxzZSwKICAgICAgICBhdXRob3JpemF0aW9u"
    "LmdldCgibmV0d29ya19hY2Nlc3NfcGVybWl0dGVkIikgaXMgbm90IEZhbHNlLAogICAgICAgIGF1"
    "dGhvcml6YXRpb24uZ2V0KCJleHRlcm5hbF9zcGVuZCIpICE9IDAsCiAgICAgICAgYXV0aG9yaXph"
    "dGlvbi5nZXQoIm1lYXN1cmVkX2V4ZWN1dGlvbl9hdXRob3JpemVkIikgaXMgbm90IEZhbHNlLAog"
    "ICAgKQopOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJhdXRob3JpemF0aW9uIHNhZmV0eSBlbnZl"
    "bG9wZSBkcmlmdGVkIikKbm93ID0gZGF0ZXRpbWUubm93KFVUQykKaXNzdWVkX2F0ID0gX3BhcnNl"
    "X3RpbWVzdGFtcChhdXRob3JpemF0aW9uLmdldCgiaXNzdWVkX2F0IikpCmV4cGlyZXNfYXQgPSBf"
    "cGFyc2VfdGltZXN0YW1wKGF1dGhvcml6YXRpb24uZ2V0KCJleHBpcmVzX2F0IikpCmlmIG5vdCBp"
    "c3N1ZWRfYXQgPD0gbm93IDwgZXhwaXJlc19hdDoKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiYXV0"
    "aG9yaXphdGlvbiBpcyBvdXRzaWRlIGl0cyB2YWxpZGl0eSB3aW5kb3ciKQoKaGFybmVzc19lbnRy"
    "aWVzID0gW2VudHJ5IGZvciBlbnRyeSBpbiBlbnRyaWVzIGlmIGVudHJ5LmdldCgicm9sZSIpID09"
    "ICJoYXJuZXNzX3NvdXJjZSJdCmlmIGxlbihoYXJuZXNzX2VudHJpZXMpICE9IDE6CiAgICByYWlz"
    "ZSBSdW50aW1lRXJyb3IoImRhdGFzZXQgbWFuaWZlc3QgbXVzdCBjb250YWluIG9uZSBoYXJuZXNz"
    "X3NvdXJjZSBlbnRyeSIpCmhhcm5lc3NfZW50cnkgPSBoYXJuZXNzX2VudHJpZXNbMF0KaWYgaGFy"
    "bmVzc19lbnRyeS5nZXQoImFydGlmYWN0X2Zvcm1hdCIpICE9ICJzb3VyY2VfdHJlZV9kaXJlY3Rv"
    "cnkiOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJoYXJuZXNzX3NvdXJjZSBtdXN0IHVzZSBzb3Vy"
    "Y2VfdHJlZV9kaXJlY3RvcnkiKQptb3VudGVkX3JlcG9fcm9vdCA9IF9ib3VuZGVkX2lucHV0X3Bh"
    "dGgoCiAgICBoYXJuZXNzX2VudHJ5LmdldCgibW91bnRlZF9wYXRoIiksCiAgICBleHBlY3RlZF9r"
    "aW5kPSJoYXJuZXNzIHNvdXJjZSIsCikKaWYgbm90IG1vdW50ZWRfcmVwb19yb290LmlzX2Rpcigp"
    "IG9yIG1vdW50ZWRfcmVwb19yb290LmlzX3N5bWxpbmsoKToKICAgIHJhaXNlIFJ1bnRpbWVFcnJv"
    "cigiaGFybmVzc19zb3VyY2UgbXVzdCByZXNvbHZlIHRvIG9uZSByZWFsIGRpcmVjdG9yeSIpCmhh"
    "cm5lc3Nfc2hhMjU2ID0gaGFybmVzc19lbnRyeS5nZXQoInNoYTI1NiIpCmlmIF9kaXJlY3Rvcnlf"
    "c2hhMjU2KG1vdW50ZWRfcmVwb19yb290KSAhPSBoYXJuZXNzX3NoYTI1NjoKICAgIHJhaXNlIFJ1"
    "bnRpbWVFcnJvcigiaGFybmVzc19zb3VyY2UgaWRlbnRpdHkgZG9lcyBub3QgbWF0Y2ggdGhlIG1h"
    "bmlmZXN0IikKbW91bnRlZF9zb3VyY2Vfcm9vdCA9IG1vdW50ZWRfcmVwb19yb290IC8gInNyYyIK"
    "cmVxdWlyZWRfaGFybmVzc19wYXRocyA9ICgKICAgIG1vdW50ZWRfcmVwb19yb290IC8gInB5cHJv"
    "amVjdC50b21sIiwKICAgIG1vdW50ZWRfc291cmNlX3Jvb3QgLyAiYXVyYWdhdGV3YXkiLAogICAg"
    "bW91bnRlZF9yZXBvX3Jvb3QgLyAiZGF0YS9ldmFscy9iZW5jaG1hcmsvZW52aXJvbm1lbnQtcXVh"
    "bGlmaWNhdGlvbi12MS8iCiAgICAicXVhbGlmaWNhdGlvbl9leGVjdXRpb25fcmVxdWVzdC5qc29u"
    "IiwKKQppZiBub3QgcmVxdWlyZWRfaGFybmVzc19wYXRoc1swXS5pc19maWxlKCk6CiAgICByYWlz"
    "ZSBSdW50aW1lRXJyb3IoImhhcm5lc3Nfc291cmNlIGRvZXMgbm90IGNvbnRhaW4gcHlwcm9qZWN0"
    "LnRvbWwiKQppZiBub3QgcmVxdWlyZWRfaGFybmVzc19wYXRoc1sxXS5pc19kaXIoKToKICAgIHJh"
    "aXNlIFJ1bnRpbWVFcnJvcigiaGFybmVzc19zb3VyY2UgZG9lcyBub3QgY29udGFpbiBzcmMvYXVy"
    "YWdhdGV3YXkiKQppZiBub3QgcmVxdWlyZWRfaGFybmVzc19wYXRoc1syXS5pc19maWxlKCk6CiAg"
    "ICByYWlzZSBSdW50aW1lRXJyb3IoImhhcm5lc3Nfc291cmNlIGRvZXMgbm90IGNvbnRhaW4gdGhl"
    "IGV4ZWN1dGlvbiByZXF1ZXN0IikKaWYgbm90IEtBR0dMRV9XT1JLSU5HX1JPT1QuaXNfZGlyKCk6"
    "CiAgICByYWlzZSBSdW50aW1lRXJyb3IoIi9rYWdnbGUvd29ya2luZyBpcyB1bmF2YWlsYWJsZSIp"
    "CmlmIE1BVEVSSUFMSVpFRF9IQVJORVNTX1JPT1QuZXhpc3RzKCk6CiAgICByYWlzZSBSdW50aW1l"
    "RXJyb3IoIndyaXRhYmxlIGhhcm5lc3MgZGVzdGluYXRpb24gYWxyZWFkeSBleGlzdHMiKQpzaHV0"
    "aWwuY29weXRyZWUobW91bnRlZF9yZXBvX3Jvb3QsIE1BVEVSSUFMSVpFRF9IQVJORVNTX1JPT1Qp"
    "CnJlcG9fcm9vdCA9IE1BVEVSSUFMSVpFRF9IQVJORVNTX1JPT1QucmVzb2x2ZSgpCmlmIF9kaXJl"
    "Y3Rvcnlfc2hhMjU2KHJlcG9fcm9vdCkgIT0gaGFybmVzc19zaGEyNTY6CiAgICByYWlzZSBSdW50"
    "aW1lRXJyb3IoIndyaXRhYmxlIGhhcm5lc3MgY29weSBpZGVudGl0eSBkcmlmdGVkIikKc291cmNl"
    "X3Jvb3QgPSByZXBvX3Jvb3QgLyAic3JjIgoKcnVudGltZV9mYWN0b3J5ID0gYXV0aG9yaXphdGlv"
    "bi5nZXQoInJ1bnRpbWVfZmFjdG9yeSIpCmlmIG5vdCBpc2luc3RhbmNlKHJ1bnRpbWVfZmFjdG9y"
    "eSwgZGljdCk6CiAgICByYWlzZSBSdW50aW1lRXJyb3IoImF1dGhvcml6YXRpb24gcnVudGltZSBm"
    "YWN0b3J5IGlzIGludmFsaWQiKQppZiBydW50aW1lX2ZhY3RvcnkuZ2V0KCJmYWN0b3J5X3BhdGgi"
    "KSAhPSBFWFBFQ1RFRF9SVU5USU1FX0ZBQ1RPUlk6CiAgICByYWlzZSBSdW50aW1lRXJyb3IoImF1"
    "dGhvcml6YXRpb24gcnVudGltZSBmYWN0b3J5IHBhdGggZHJpZnRlZCIpCmFkYXB0ZXJfcmVsYXRp"
    "dmVfcGF0aCA9IFBhdGgocnVudGltZV9mYWN0b3J5LmdldCgiYXJ0aWZhY3RfcGF0aCIsICIiKSkK"
    "aWYgYWRhcHRlcl9yZWxhdGl2ZV9wYXRoLmlzX2Fic29sdXRlKCkgb3IgIi4uIiBpbiBhZGFwdGVy"
    "X3JlbGF0aXZlX3BhdGgucGFydHM6CiAgICByYWlzZSBSdW50aW1lRXJyb3IoInJ1bnRpbWUgYWRh"
    "cHRlciBwYXRoIG11c3QgYmUgcmVwb3NpdG9yeS1yZWxhdGl2ZSIpCmFkYXB0ZXJfcGF0aCA9IChy"
    "ZXBvX3Jvb3QgLyBhZGFwdGVyX3JlbGF0aXZlX3BhdGgpLnJlc29sdmUoKQppZiBhZGFwdGVyX3Bh"
    "dGggIT0gcmVwb19yb290IGFuZCByZXBvX3Jvb3Qgbm90IGluIGFkYXB0ZXJfcGF0aC5wYXJlbnRz"
    "OgogICAgcmFpc2UgUnVudGltZUVycm9yKCJydW50aW1lIGFkYXB0ZXIgbXVzdCByZW1haW4gaW5z"
    "aWRlIGhhcm5lc3Nfc291cmNlIikKaWYgX2ZpbGVfc2hhMjU2KGFkYXB0ZXJfcGF0aCkgIT0gcnVu"
    "dGltZV9mYWN0b3J5LmdldCgiYXJ0aWZhY3Rfc2hhMjU2Iik6CiAgICByYWlzZSBSdW50aW1lRXJy"
    "b3IoInJ1bnRpbWUgYWRhcHRlciBpZGVudGl0eSBkb2VzIG5vdCBtYXRjaCBhdXRob3JpemF0aW9u"
    "IikKCm9zLmVudmlyb25bIkFVUkFHQVRFV0FZX1JFUE9fUk9PVCJdID0gc3RyKHJlcG9fcm9vdCkK"
    "c3lzLnBhdGguaW5zZXJ0KDAsIHN0cihzb3VyY2Vfcm9vdCkpCgpleGVjdXRpb25fbW9kdWxlID0g"
    "aW1wb3J0bGliLmltcG9ydF9tb2R1bGUoCiAgICAiYXVyYWdhdGV3YXkubG9jYWxfYWJjLmZ1bGxf"
    "YWJjX2xvY2FsX2Vudmlyb25tZW50X3F1YWxpZmljYXRpb25fZXhlY3V0aW9uIgopCnN1bW1hcnkg"
    "PSBleGVjdXRpb25fbW9kdWxlLmV4ZWN1dGVfZnJvbV9lbnZpcm9ubWVudCgpCnN1bW1hcnk="
)

EXPECTED_HARNESS_SOURCE = Path(
    "/kaggle/input/datasets/kabomolefe/auragateway-qualification-harness-4dfd799-"
    "v1"
)
EXPECTED_MODEL_SNAPSHOT = Path(
    "/kaggle/input/datasets/kabomolefe/auragateway-qwen2-5-0-5b-offline-v1/auraga"
    "teway-qwen2.5-0.5b-instruct-7ae557604adf67be50417f59c2c2f167def9a775/hf_home"
    "/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c"
    "2c2f167def9a775"
)
EXPECTED_VLLM_WHEEL = Path(
    "/kaggle/input/notebooks/kabomolefe/auragateway-vllm-wheel-recovery-v1/auraga"
    "teway_vllm_wheels_v1/vllm-0.25.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
)

MINIMUM_LAUNCH_WINDOW_MINUTES = 120
MAXIMUM_EVIDENCE_ZIP_BYTES = 2097152

RUNTIME_EVIDENCE_PATHS = (
    "data/evals/benchmark/environment-qualification-v1/cache_metric_capability_report.json",
    "data/evals/benchmark/environment-qualification-v1/gpu_topology_report.json",
    "data/evals/benchmark/environment-qualification-v1/kaggle_runtime_dependency_lock.json",
    "data/evals/benchmark/environment-qualification-v1/manifest.json",
    "data/evals/benchmark/environment-qualification-v1/model_identity_report.json",
    "data/evals/benchmark/environment-qualification-v1/qualification_report.json",
    "data/evals/benchmark/environment-qualification-v1/reset_capability_report.json",
    "data/evals/benchmark/environment-qualification-v1/worker_health_report.json",
)

stage = "launcher_initialization"


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def parse_timestamp(raw_value: object) -> datetime:
    if not isinstance(raw_value, str):
        raise RuntimeError("authorization timestamp is invalid")
    value = datetime.fromisoformat(raw_value.replace("Z", "+00:00"))
    if value.tzinfo is None or value.utcoffset() is None:
        raise RuntimeError("authorization timestamp must be timezone-aware")
    return value.astimezone(UTC)


def port_open(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1.0):
            return True
    except OSError:
        return False


def exact_candidates(filename: str) -> tuple[Path, ...]:
    return tuple(
        sorted(
            (
                path.resolve()
                for path in INPUT_ROOT.rglob(filename)
                if path.is_file() and not path.is_symlink()
            ),
            key=lambda item: item.as_posix(),
        )
    )


def validate_regular_file(path: Path, *, maximum_bytes: int) -> None:
    if not path.is_file() or path.is_symlink():
        raise RuntimeError(f"expected one regular file: {path.name}")
    if path.stat().st_size > maximum_bytes:
        raise RuntimeError(f"bounded file exceeds its size limit: {path.name}")


def write_failure_bundle(error: BaseException) -> None:
    evidence_found: list[str] = []
    if MATERIALIZED_HARNESS_ROOT.is_dir():
        for relative_path in RUNTIME_EVIDENCE_PATHS:
            path = MATERIALIZED_HARNESS_ROOT / relative_path
            if path.is_file() and not path.is_symlink():
                evidence_found.append(relative_path)

    failure = {
        "schema_version": "1.0.0",
        "status": "FAILED",
        "stage": stage,
        "captured_at": datetime.now(UTC).replace(microsecond=0).isoformat(),
        "exception_type": type(error).__name__,
        "safe_message": str(error)[:1000],
        "runtime_evidence_found": sorted(evidence_found),
        "ports_open": [
            port
            for port in (8001, 8002)
            if port_open(port)
        ],
        "benchmark_trajectory_requests_permitted": 0,
        "customer_data_used": False,
        "credentials_used": False,
        "provider_calls_performed": False,
        "external_spend": 0,
    }
    trace = "".join(
        traceback.format_exception(type(error), error, error.__traceback__)
    )[-65536:]

    if EVIDENCE_ZIP_PATH.exists():
        EVIDENCE_ZIP_PATH.unlink()

    with zipfile.ZipFile(
        EVIDENCE_ZIP_PATH,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        archive.writestr(
            "launcher_failure.json",
            canonical_json(failure),
        )
        archive.writestr(
            "launcher_failure_trace.txt",
            trace,
        )

    if EVIDENCE_ZIP_PATH.stat().st_size > MAXIMUM_EVIDENCE_ZIP_BYTES:
        EVIDENCE_ZIP_PATH.unlink(missing_ok=True)
        raise RuntimeError("failure evidence ZIP exceeded the size budget")

    print(f"artifact={EVIDENCE_ZIP_PATH}")
    print(f"size_bytes={EVIDENCE_ZIP_PATH.stat().st_size}")
    print(f"sha256={file_sha256(EVIDENCE_ZIP_PATH)}")
    print("qualification_status=FAILED")
    print("upload_only_this_file=true")


try:
    stage = "fresh_session_guard"

    if not INPUT_ROOT.is_dir() or not WORK_ROOT.is_dir():
        raise RuntimeError("required Kaggle filesystem roots are unavailable")

    if len(NOTEBOOK_NAME) > 50:
        raise RuntimeError("Kaggle notebook name exceeds 50 characters")

    loaded_runtime_modules = sorted(
        name
        for name in sys.modules
        if name == "vllm"
        or name.startswith("vllm.")
        or name == "transformers"
        or name.startswith("transformers.")
    )
    if loaded_runtime_modules:
        raise RuntimeError(
            "qualification requires a fresh kernel without loaded runtime modules: "
            + ", ".join(loaded_runtime_modules)
        )

    if "torch" in sys.modules:
        torch_module = sys.modules["torch"]
        cuda_module = getattr(torch_module, "cuda", None)
        is_initialized = getattr(cuda_module, "is_initialized", None)
        if callable(is_initialized) and bool(is_initialized()):
            raise RuntimeError("qualification found a pre-existing CUDA context")

    stale_runtime_keys = sorted(
        key
        for key in os.environ
        if key.startswith("AURAGATEWAY_") or key.startswith("VLLM_")
    )
    if stale_runtime_keys:
        raise RuntimeError(
            "qualification found stale AuraGateway/vLLM environment variables: "
            + ", ".join(stale_runtime_keys)
        )

    if MATERIALIZED_HARNESS_ROOT.exists():
        raise RuntimeError("writable harness destination already exists")

    if EVIDENCE_ZIP_PATH.exists():
        raise RuntimeError("qualification evidence ZIP already exists")

    open_ports = [port for port in (8001, 8002) if port_open(port)]
    if open_ports:
        raise RuntimeError(f"vLLM worker ports are already open: {open_ports}")

    for expected_path in (
        EXPECTED_HARNESS_SOURCE,
        EXPECTED_MODEL_SNAPSHOT,
        EXPECTED_VLLM_WHEEL,
    ):
        resolved = expected_path.resolve()
        if INPUT_ROOT not in resolved.parents:
            raise RuntimeError("a static input escaped /kaggle/input")
        if not resolved.exists() or resolved.is_symlink():
            raise RuntimeError(f"expected static input is unavailable: {resolved}")

    stage = "control_output_discovery"

    authorization_candidates = exact_candidates(AUTHORIZATION_FILENAME)
    manifest_candidates = exact_candidates(DATASET_MANIFEST_FILENAME)
    control_manifest_candidates = exact_candidates(CONTROL_MANIFEST_NAME)
    receipt_candidates = exact_candidates(CONTROL_RECEIPT_NAME)

    candidate_sets = (
        authorization_candidates,
        manifest_candidates,
        control_manifest_candidates,
        receipt_candidates,
    )
    if any(len(candidates) != 1 for candidates in candidate_sets):
        raise RuntimeError(
            "control output must expose exactly one authorization, manifest, "
            "control manifest, and receipt"
        )

    authorization_path = authorization_candidates[0]
    dataset_manifest_path = manifest_candidates[0]
    control_manifest_path = control_manifest_candidates[0]
    receipt_path = receipt_candidates[0]

    control_root = authorization_path.parent
    if any(path.parent != control_root for path in (
        dataset_manifest_path,
        control_manifest_path,
        receipt_path,
    )):
        raise RuntimeError("control output files must share one flat directory")

    if CONTROL_NOTEBOOK_TOKEN not in control_root.as_posix():
        raise RuntimeError("control output did not originate from the governed notebook")

    expected_control_names = {
        AUTHORIZATION_FILENAME,
        DATASET_MANIFEST_FILENAME,
        CONTROL_MANIFEST_NAME,
        CONTROL_RECEIPT_NAME,
    }
    observed_control_names = {
        path.name
        for path in control_root.iterdir()
        if path.is_file()
    }
    if observed_control_names != expected_control_names:
        raise RuntimeError("control output file set drifted")

    for path in control_root.iterdir():
        if path.is_symlink() or not path.is_file():
            raise RuntimeError("control output contains an unsafe member type")
        if path.suffix.lower() in {".zip", ".tar", ".tgz", ".7z"}:
            raise RuntimeError("control output cannot contain nested archives")
        validate_regular_file(path, maximum_bytes=1024 * 1024)

    control_manifest = json.loads(
        control_manifest_path.read_text(encoding="utf-8")
    )
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    authorization = json.loads(
        authorization_path.read_text(encoding="utf-8")
    )
    dataset_manifest = json.loads(
        dataset_manifest_path.read_text(encoding="utf-8")
    )
    if not all(
        isinstance(payload, dict)
        for payload in (
            control_manifest,
            receipt,
            authorization,
            dataset_manifest,
        )
    ):
        raise RuntimeError("control output JSON roots must be objects")

    if control_manifest.get("source_main_merge_commit") != (
        EXPECTED_SOURCE_MAIN_MERGE_COMMIT
    ):
        raise RuntimeError("control output source main binding drifted")
    if control_manifest.get("authorization_source_main_merge_commit") != (
        EXPECTED_AUTHORIZATION_SOURCE_MAIN_MERGE_COMMIT
    ):
        raise RuntimeError("authorization source main binding drifted")

    if control_manifest.get("authorization_file_sha256") != file_sha256(
        authorization_path
    ):
        raise RuntimeError("authorization file identity drifted")
    if control_manifest.get("dataset_manifest_file_sha256") != file_sha256(
        dataset_manifest_path
    ):
        raise RuntimeError("dataset manifest file identity drifted")

    if receipt.get("control_manifest_sha256") != file_sha256(
        control_manifest_path
    ):
        raise RuntimeError("control materialization receipt drifted")

    if authorization.get("source_main_merge_commit") != (
        EXPECTED_AUTHORIZATION_SOURCE_MAIN_MERGE_COMMIT
    ):
        raise RuntimeError("authorization merge binding drifted")
    if authorization.get("decision") != "AUTHORIZED":
        raise RuntimeError("authorization decision is not AUTHORIZED")

    if any(
        (
            authorization.get("maximum_workers") != 2,
            authorization.get("maximum_kaggle_sessions") != 1,
            authorization.get("maximum_model_requests") != 8,
            authorization.get("maximum_output_tokens_per_request") != 32,
            authorization.get("benchmark_trajectory_requests_permitted") != 0,
            authorization.get("network_access_permitted") is not False,
            authorization.get("credentials_permitted") is not False,
            authorization.get("customer_data_permitted") is not False,
            authorization.get("external_spend") != 0,
            authorization.get("measured_execution_authorized") is not False,
        )
    ):
        raise RuntimeError("authorization safety envelope drifted")

    expires_at = parse_timestamp(authorization.get("expires_at"))
    remaining_minutes = int(
        (expires_at - datetime.now(UTC)).total_seconds() // 60
    )
    if remaining_minutes < MINIMUM_LAUNCH_WINDOW_MINUTES:
        raise RuntimeError(
            "authorization has insufficient time remaining for a cold launch"
        )

    entries = dataset_manifest.get("entries")
    if not isinstance(entries, list) or len(entries) != 3:
        raise RuntimeError("dataset manifest entry set drifted")
    observed_mounts = {
        str(entry.get("role")): str(entry.get("mounted_path"))
        for entry in entries
        if isinstance(entry, dict)
    }
    expected_mounts = {
        "harness_source": EXPECTED_HARNESS_SOURCE.as_posix(),
        "model_artifacts": EXPECTED_MODEL_SNAPSHOT.as_posix(),
        "vllm_wheel": EXPECTED_VLLM_WHEEL.as_posix(),
    }
    if observed_mounts != expected_mounts:
        raise RuntimeError("dataset manifest mount bindings drifted")

    stage = "reviewed_core_execution"

    reviewed_core = base64.b64decode(
        REVIEWED_CORE_B64.encode("ascii"),
        validate=True,
    )
    if sha256_bytes(reviewed_core) != EXPECTED_REVIEWED_CORE_SHA256:
        raise RuntimeError("reviewed qualification core identity drifted")

    os.environ["AURAGATEWAY_QUALIFICATION_AUTHORIZATION"] = str(
        authorization_path
    )
    os.environ["AURAGATEWAY_QUALIFICATION_DATASET_MANIFEST"] = str(
        dataset_manifest_path
    )

    execution_namespace: dict[str, object] = {}
    exec(
        compile(
            reviewed_core.decode("utf-8"),
            "auragateway_reviewed_qualification_core_v1.py",
            "exec",
        ),
        execution_namespace,
    )

    summary = execution_namespace.get("summary")
    if not isinstance(summary, dict):
        raise RuntimeError("qualification execution did not return a summary")

    expected_summary = {
        "model_request_count": 6,
        "runtime_evidence_generated": True,
        "runtime_evidence_count": 8,
        "environment_qualified": True,
        "measured_execution_authorized": False,
        "external_spend": 0,
        "next_gate": (
            "full_abc_local_full_run_environment_qualification_evidence_review"
        ),
    }
    for key, expected in expected_summary.items():
        if summary.get(key) != expected:
            raise RuntimeError(
                f"qualification summary drifted: {key}"
            )

    stage = "success_evidence_packaging"

    repo_root_raw = os.environ.get("AURAGATEWAY_REPO_ROOT")
    if repo_root_raw is None:
        raise RuntimeError("reviewed core did not bind AURAGATEWAY_REPO_ROOT")
    repo_root = Path(repo_root_raw).resolve()
    if repo_root != MATERIALIZED_HARNESS_ROOT.resolve():
        raise RuntimeError("reviewed core repository root drifted")

    evidence_files: list[tuple[str, Path]] = []
    evidence_checksums: dict[str, dict[str, object]] = {}
    for relative_path in RUNTIME_EVIDENCE_PATHS:
        path = repo_root / relative_path
        validate_regular_file(path, maximum_bytes=512 * 1024)
        evidence_files.append((Path(relative_path).name, path))
        evidence_checksums[Path(relative_path).name] = {
            "sha256": file_sha256(path),
            "size_bytes": path.stat().st_size,
        }

    launcher_summary = {
        "schema_version": "1.0.0",
        "status": "QUALIFIED",
        "notebook_name": NOTEBOOK_NAME,
        "captured_at": datetime.now(UTC).replace(microsecond=0).isoformat(),
        "reviewed_core_sha256": EXPECTED_REVIEWED_CORE_SHA256,
        "source_main_merge_commit": EXPECTED_SOURCE_MAIN_MERGE_COMMIT,
        "authorization_source_main_merge_commit": (
            EXPECTED_AUTHORIZATION_SOURCE_MAIN_MERGE_COMMIT
        ),
        "authorization_file_sha256": file_sha256(authorization_path),
        "dataset_manifest_file_sha256": file_sha256(dataset_manifest_path),
        "remaining_minutes_at_launch": remaining_minutes,
        "summary": summary,
        "benchmark_trajectory_requests_permitted": 0,
        "customer_data_used": False,
        "credentials_used": False,
        "provider_calls_performed": False,
        "external_spend": 0,
    }

    if EVIDENCE_ZIP_PATH.exists():
        raise RuntimeError("qualification evidence ZIP appeared unexpectedly")

    with zipfile.ZipFile(
        EVIDENCE_ZIP_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for archive_name, path in sorted(evidence_files):
            archive.write(path, arcname=archive_name)
        archive.writestr(
            "evidence_bundle_sha256.json",
            canonical_json(evidence_checksums),
        )
        archive.writestr(
            "launcher_summary.json",
            canonical_json(launcher_summary),
        )

    zip_size = EVIDENCE_ZIP_PATH.stat().st_size
    if zip_size > MAXIMUM_EVIDENCE_ZIP_BYTES:
        EVIDENCE_ZIP_PATH.unlink(missing_ok=True)
        raise RuntimeError("qualification evidence ZIP exceeded the size budget")

    print(f"artifact={EVIDENCE_ZIP_PATH}")
    print(f"size_bytes={zip_size}")
    print(f"sha256={file_sha256(EVIDENCE_ZIP_PATH)}")
    print("qualification_status=QUALIFIED")
    print("runtime_evidence_count=8")
    print("model_request_count=6")
    print("benchmark_trajectory_requests_permitted=0")
    print("upload_only_this_file=true")

except BaseException as error:
    write_failure_bundle(error)
    raise
